<p style="float: left;"><a href="by-name-parameters.ipynb" target="_blank">Previous</a></p>
<p style="float: right;"><a href="packages-and-imports.ipynb" target="_blank">Next</a></p>
<p style="text-align:center;">Tour of Scala</p>
<div style="clear: both;"></div>

# Annotations

Annotations associate meta-information with definitions. For example, the annotation `@deprecated` before a method causes the compiler to print a warning if the method is used.

In [1]:
object DeprecationDemo {
  @deprecated("deprecation message", "release # which deprecates method")
  def hello = "hola"
}

DeprecationDemo.hello

cmd1.sc:6: method hello in object DeprecationDemo is deprecated (since release # which deprecates method): deprecation message
val res1_1 = DeprecationDemo.hello
                             ^


defined object DeprecationDemo
res1_1: String = "hola"

Calling `hello` still compiles and runs (it returns `"hola"`), but the compiler prints a deprecation warning with the message given in the annotation.

An annotation clause applies to the first definition or declaration following it. More than one annotation clause may precede a definition and declaration. The order in which these clauses are given does not matter.


## Annotations that ensure correctness of encodings
Certain annotations will actually cause compilation to fail if a condition(s) is not met. For example, the annotation `@tailrec` ensures that a method is [tail-recursive](https://en.wikipedia.org/wiki/Tail_call). Tail-recursion can keep memory requirements constant. Here's how it's used in a method which calculates the factorial:

In [2]:
import scala.annotation.tailrec

def factorial(x: Int): Int = {

  @tailrec
  def factorialHelper(x: Int, accumulator: Int): Int = {
    if (x == 1) accumulator else factorialHelper(x - 1, accumulator * x)
  }
  factorialHelper(x, 1)
}

import scala.annotation.tailrec
defined function factorial

The `factorialHelper` method has the `@tailrec` which ensures the method is indeed tail-recursive. If we were to change the implementation of `factorialHelper` to the following, it would fail:

In [3]:
import scala.annotation.tailrec

def factorial(x: Int): Int = {
  @tailrec
  def factorialHelper(x: Int): Int = {
    if (x == 1) 1 else x * factorialHelper(x - 1)
  }
  factorialHelper(x)
}

cmd3.sc:6: could not optimize @tailrec annotated method factorialHelper: it contains a recursive call not in tail position
    if (x == 1) 1 else x * factorialHelper(x - 1)
                                          ^
Compilation Failed

: Compilation Failed

We would get the message "Recursive call not in tail position".

## Annotations affecting code generation


Some annotations like `@inline` affect the generated code (i.e. your jar file might have different bytes than if you hadn't used the annotation). Inlining means inserting the code in a method's body at the call site. The resulting bytecode is longer, but hopefully runs faster. Using the annotation `@inline` does not ensure that a method will be inlined, but it will cause the compiler to do it if and only if some heuristics about the size of the generated code are met.


### Java Annotations ###
When writing Scala code which interoperates with Java, there are a few differences in annotation syntax to note.
**Note:** Make sure you use the `-target:jvm-1.8` option with Java annotations.

Java has user-defined metadata in the form of [annotations](https://docs.oracle.com/javase/tutorial/java/annotations/). A key feature of annotations is that they rely on specifying name-value pairs to initialize their elements. For instance, if we need an annotation to track the source of some class we might define it as

```java
@interface Source {
  public String url();
  public String mail();
}
```

And then apply it as follows

```java
@Source(url = "https://coders.com/",
        mail = "support@coders.com")
public class MyJavaClass extends TheirClass ...
```

An annotation application in Scala looks like a constructor invocation, but to instantiate a Java annotation one has to use named arguments:

```scala
@Source(url = "https://coders.com/",
        mail = "support@coders.com")
class MyScalaClass ...
```

This syntax is quite tedious if the annotation contains only one element (without default value) so, by convention, if the name is specified as `value` it can be applied in Java using a constructor-like syntax:

```java
@interface SourceURL {
    public String value();
    public String mail() default "";
}
```

And then apply it as follows:

```java
@SourceURL("https://coders.com/")
public class MyJavaClass extends TheirClass ...
```

In this case, Scala provides the same possibility:

```scala
@SourceURL("https://coders.com/")
class MyScalaClass ...
```

The `mail` element was specified with a default value so we need not explicitly provide a value for it.
However, if we need to provide one then in Java we must also explicitly name the `value` parameter:

```java
@SourceURL(value = "https://coders.com/",
           mail = "support@coders.com")
public class MyJavaClass extends TheirClass ...
```

Scala provides more flexibility in this respect, so we can choose to only name the `mail` parameter:

```scala
@SourceURL("https://coders.com/",
           mail = "support@coders.com")
class MyScalaClass ...
```

<p style="float: left;"><a href="by-name-parameters.ipynb" target="_blank">Previous</a></p>
<p style="float: right;"><a href="packages-and-imports.ipynb" target="_blank">Next</a></p>
<p style="text-align:center;">Tour of Scala</p>
<div style="clear: both;"></div>